# DL_DOA — No-Clone, Step-by-Step Reproduction Notebook

Ei notebook-e **kono `git clone` nei**. Tumi already `dldoa_dataset_generation.py --mode save_all` diye
dataset generate kore rekhecho — ei notebook shudhu shei already-generated dataset file gulo load kore,
tarpor **protiti stage (load -> explore -> split -> model -> train -> test -> metrics -> plot) alada alada
cell-e** bhaga kora, jate protiti step-er output age dekha jay ebong dorkar hole shei cell-ta shudhu
modify kore abar run kora jay.

## Ei notebook chalanor age tomar kache ja thakte hobe

`dldoa_dataset_generation.py --mode save_all --output_dir dataset` run korle ei file gulo toiri hoy —
egulo Colab-e upload/drive-mount kore rakho (niche Part 1-e bola ache kivabe):

```
dataset/train_data.npz      dataset/train_gt.npz
dataset/val_data.npz        dataset/val_gt.npz        dataset/val_meta.npz        dataset/val_features.npy
dataset/test_data.npz       dataset/test_gt.npz       dataset/test_meta.npz       dataset/test_features.npy
```

(Jodi tumi shudhu `save_train` / `save_val` / `save_test` alada alada run korey thako tao shomossha nei —
niche jei file na paoa jabe shei part shudhu skip hobe.)

Optionally, jodi pretrained weight (`inf_model_007_256_resnet.h5`, `inf_model_007_256_unet.h5`) o upload
koro, tahole training skip kore shorashori testing-e jete parbe (Part 10).

## Part 0 — Setup: dependencies install

Colab-e already **tensorflow, numpy, scipy, matplotlib, scikit-learn, tqdm, opencv** shob preinstalled
thake, ar egulo ekta-r shathe ekta compatible-vabe build kora (same ABI/wheel set)। Age ei notebook
`tensorflow==2.21.0 numpy==2.4.6 ...` er moto exact version force-install korto — eta-i asol
error-er karon hote pare, karon:

- Colab-er already-loaded numpy/tensorflow-er shathe version conflict lagle pip resolver error dey, **অথবা**
- install shoja succeed kore, kintu already-imported/pre-linked library-gulo notun version-er shathe
  match kore na (ABI mismatch) — jar fole porer cell-e (Part 0.1 Imports) hঠাৎ error ashe, jotokkhon na
  tumi **Runtime > Restart runtime** kore abar shuru koro।

Tai ekhon shudhu shei package-gulo install kora hocche jegulo shotti-i missing — already thakle kichu
touch kora hobe na, tai kono version-conflict/restart lagbe na।

In [ ]:
import importlib, subprocess, sys

def ensure(pip_name, import_name=None):
    import_name = import_name or pip_name
    try:
        importlib.import_module(import_name)
        print(f'{import_name}: already available, skip.')
    except ImportError:
        print(f'{import_name}: not found, installing {pip_name} ...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pip_name], check=True)

# tensorflow / numpy / scipy / matplotlib / scikit-learn / tqdm -- Colab-e already thake,
# eigulo re-pin kora hocche na (age-er version-conflict-er karon eta-i chilo).
for pip_name, import_name in [
    ('opencv-python-headless', 'cv2'),
]:
    ensure(pip_name, import_name)

print('Dependency check done.')

### Part 0.1 — Imports

In [ ]:
import os
import itertools
import pickle

import numpy as np
import scipy.ndimage
from scipy.optimize import linear_sum_assignment
import matplotlib.pyplot as plt
import cv2
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import tensorflow as tf
from tensorflow.keras.layers import (
    Dense, Conv2D, Input, BatchNormalization, MaxPooling2D,
    Flatten, Activation, Add, Conv2DTranspose,
)
from tensorflow.keras import Model
from tensorflow.keras.optimizers import Adam, RMSprop

print('TensorFlow:', tf.__version__)
print('Keras:', tf.keras.__version__ if hasattr(tf.keras, '__version__') else '?')
print('GPU available:', tf.config.list_physical_devices('GPU'))

In [ ]:
np.random.seed(42)
tf.random.set_seed(42)

## Part 1 — Tomar dataset-er path auto-detect kori

Kaggle-er "Input" panel-e dekha path (ja UI-te dekhায়) ar notebook-er ভেতর-er real mount path prayoi
alada hoy (shadharonoto real path `/kaggle/input/<dataset-slug>/...` — "datasets/<owner>/" prefix ছাড়া) —
tai hardcoded path guess kora bhul-prone. Erchaite, ei cell-ta `/kaggle/input` (ba Colab/local hole
current directory) er niche shob subfolder recursively search kore `train_data.npz` (ba `val_data.npz` /
`test_data.npz`) khunje shei exact folder-take `DATA_DIR` set kore dey — kono path guess korte hobe na।

In [ ]:
def find_dataset_dir(search_roots, markers=('train_data.npz', 'val_data.npz', 'test_data.npz')):
    for root in search_roots:
        if not os.path.isdir(root):
            continue
        for dirpath, dirnames, filenames in os.walk(root):
            if any(m in filenames for m in markers):
                return dirpath
    return None

SEARCH_ROOTS = ['/kaggle/input', '/content', '.']  # Kaggle, Colab, local -- shob jaygay check kore

DATA_DIR = find_dataset_dir(SEARCH_ROOTS)

if DATA_DIR is None:
    # Auto-detect fail korle, nijei exact path boshao (Kaggle notebook-e ei cell-er age
    # ekta cell-e `!find /kaggle/input -name train_data.npz` chaliye real path dekhe nite paro).
    DATA_DIR = '/kaggle/input/dl-doa/content/DL_DOA_CLONE/dataset'  # <-- fallback, dorkar hole change koro

assert os.path.isdir(DATA_DIR) and any(
    os.path.exists(os.path.join(DATA_DIR, m)) for m in ('train_data.npz', 'val_data.npz', 'test_data.npz')
), (
    f'DATA_DIR-e (found: {DATA_DIR}) train_data.npz/val_data.npz/test_data.npz kono-ta-i paoa jayni. '
    'Kaggle-e ekta notun cell-e `!find /kaggle/input -name "train_data.npz"` chalao, jei path ashbe '
    'shei-tar parent folder-take upore DATA_DIR-er fallback line-e boshao.'
)

MODEL_DIR = DATA_DIR  # pretrained .h5 file (thakle) o eikhane khoja hobe; alada folder hole change koro

print('DATA_DIR =', DATA_DIR)
print('Files found:')
for f in sorted(os.listdir(DATA_DIR)):
    print(' -', f)

## Part 2 — Dataset arrays load koro (`.npz` / `.npy` theke)

### Part 2.1 — Training set

In [ ]:
def _npz_path(*candidate_names):
    """First matching file among candidate_names (handles e.g. val_features.npy
    vs val_features.pkl -- different runs/tools can save this ragged array either way)."""
    for name in candidate_names:
        p = os.path.join(DATA_DIR, name)
        if os.path.exists(p):
            return p
    return None

def _load_features(path):
    if path is None:
        return None
    if path.endswith('.pkl'):
        with open(path, 'rb') as f:
            return pickle.load(f)
    return np.load(path, allow_pickle=True)

train_data_path = _npz_path('train_data.npz')
train_gt_path = _npz_path('train_gt.npz')

if train_data_path and train_gt_path:
    X_train_full = np.load(train_data_path)['data']
    Y_train_full = np.load(train_gt_path)['data']
    print('X_train_full:', X_train_full.shape, X_train_full.dtype)
    print('Y_train_full:', Y_train_full.shape, Y_train_full.dtype)
else:
    X_train_full, Y_train_full = None, None
    print('train_data.npz / train_gt.npz paoa jayni — Part 4/9 (split/training) skip hobe.')

### Part 2.2 — Validation set (paper: 1000 fixed samples, seed=42)

In [ ]:
val_data_path = _npz_path('val_data.npz')
val_gt_path = _npz_path('val_gt.npz')
val_meta_path = _npz_path('val_meta.npz')
val_features_path = _npz_path('val_features.npy', 'val_features.pkl')

if val_data_path and val_gt_path:
    X_val = np.load(val_data_path)['data']
    Y_val = np.load(val_gt_path)['data']
    meta_val = np.load(val_meta_path)['data'] if val_meta_path else None
    feat_val = _load_features(val_features_path)
    print('X_val:', X_val.shape)
    print('Y_val:', Y_val.shape)
    print('meta_val:', None if meta_val is None else meta_val.shape)
    print('feat_val entries:', None if feat_val is None else len(feat_val))
else:
    X_val, Y_val, meta_val, feat_val = None, None, None, None
    print('val_data.npz / val_gt.npz paoa jayni.')

### Part 2.3 — Test set (paper Figs. 5-6: L=3, 8 SNR points x 1000 samples)

In [ ]:
test_data_path = _npz_path('test_data.npz')
test_gt_path = _npz_path('test_gt.npz')
test_meta_path = _npz_path('test_meta.npz')
test_features_path = _npz_path('test_features.npy', 'test_features.pkl')

if test_data_path and test_gt_path:
    X_test = np.load(test_data_path)['data']
    Y_test = np.load(test_gt_path)['data']
    meta_test = np.load(test_meta_path)['data'] if test_meta_path else None
    feat_test = _load_features(test_features_path)
    print('X_test:', X_test.shape)
    print('Y_test:', Y_test.shape)
    print('meta_test:', None if meta_test is None else meta_test.shape)
    print('feat_test entries:', None if feat_test is None else len(feat_test))
else:
    X_test, Y_test, meta_test, feat_test = None, None, None, None
    print('test_data.npz / test_gt.npz paoa jayni — Part 13 (full evaluation) skip hobe.')

## Part 3 — Data exploration

In [ ]:
def summarize(name, arr):
    if arr is None:
        print(f'{name}: not loaded')
        return
    print(f'{name}: shape={arr.shape} dtype={arr.dtype} '
          f'min={np.min(arr):.4f} max={np.max(arr):.4f} mean={np.mean(arr):.4f}')

summarize('X_train_full', X_train_full)
summarize('Y_train_full', Y_train_full)
summarize('X_val', X_val)
summarize('Y_val', Y_val)
summarize('X_test', X_test)
summarize('Y_test', Y_test)

### Part 3.1 — Ekta example sample visually dekho (input real/imag + ground truth heatmap)

In [ ]:
def show_example(X, Y, idx=0, title_prefix=''):
    fig, axs = plt.subplots(1, 3, figsize=(13, 4))
    axs[0].imshow(X[idx, :, :, 0], cmap='viridis')
    axs[0].set_title(f'{title_prefix} Input — Real part')
    axs[1].imshow(X[idx, :, :, 1], cmap='viridis')
    axs[1].set_title(f'{title_prefix} Input — Imag part')
    axs[2].imshow(Y[idx, :, :, 0], cmap='hot')
    axs[2].set_title(f'{title_prefix} Ground truth heatmap')
    for ax in axs:
        ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout()
    plt.show()

if X_train_full is not None:
    show_example(X_train_full, Y_train_full, idx=0, title_prefix='[Train]')
elif X_val is not None:
    show_example(X_val, Y_val, idx=0, title_prefix='[Val]')

### Part 3.2 — Validation set-er condition distribution (L, SNR, P) dekho

In [ ]:
if meta_val is not None:
    L_vals = meta_val[:, 0]
    SNR_vals = meta_val[:, 1]
    fig, axs = plt.subplots(1, 2, figsize=(11, 4))
    axs[0].hist(L_vals, bins=np.arange(1, 11) - 0.5, rwidth=0.8)
    axs[0].set_title('L (number of paths) distribution — validation set')
    axs[0].set_xlabel('L')
    axs[1].hist(SNR_vals, bins=20)
    axs[1].set_title('SNR (dB) distribution — validation set')
    axs[1].set_xlabel('SNR (dB)')
    plt.tight_layout()
    plt.show()
else:
    print('meta_val paoa jayni, distribution plot skip.')

## Part 4 — Train / (extra) validation split

`val_data.npz` already paper-er official fixed 1000-sample validation set (seed=42) — eta already
train theke completely alada. Ei cell-ta **extra**: jodi tumi training set-er modhkhe theke arek-ta
choto held-out subset (quick sanity check-er jonno) chao, seta ei split diye toiri hoy.
`X_train`, `Y_train` ei notebook-er baki shob training cell-e use hobe.

In [ ]:
HOLDOUT_FRACTION = 0.1  # 10% training data ke internal holdout hishebe rakha hocche; 0 dile pura train set-i thakbe

if X_train_full is not None:
    if HOLDOUT_FRACTION > 0:
        X_train, X_train_holdout, Y_train, Y_train_holdout = train_test_split(
            X_train_full, Y_train_full, test_size=HOLDOUT_FRACTION, random_state=42,
        )
    else:
        X_train, Y_train = X_train_full, Y_train_full
        X_train_holdout, Y_train_holdout = None, None

    print('X_train:', X_train.shape)
    print('Y_train:', Y_train.shape)
    if X_train_holdout is not None:
        print('X_train_holdout:', X_train_holdout.shape)
else:
    X_train = Y_train = X_train_holdout = Y_train_holdout = None

## Part 5 — Model architecture: UNet (building blocks)

Paper-er `tvt_models.py`-r UNet architecture, protita helper function-o alada cell — jate dorkar hole
ekta building block shudhu edit kore abar model rebuild kora jay.

In [ ]:
def Conv_Block(inputs, model_width, kernel, multiplier):
    """Conv2D -> BatchNorm -> ReLU"""
    x = tf.keras.layers.Conv2D(model_width * multiplier, kernel, padding='same')(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    return x

In [ ]:
def trans_conv2D(inputs, model_width, multiplier):
    """Transposed-conv upsampling -> BatchNorm -> ReLU"""
    x = tf.keras.layers.Conv2DTranspose(model_width * multiplier, (2, 2), strides=(2, 2), padding='same')(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    return x

In [ ]:
def Concat_Block(input1, *argv):
    """Concatenate multiple tensors along the channel axis"""
    cat = input1
    for arg in argv:
        cat = tf.keras.layers.concatenate([cat, arg], axis=-1)
    return cat

In [ ]:
def Feature_Extraction_Block(inputs, model_width, feature_number):
    """Optional autoencoder-style latent bottleneck"""
    shape = inputs.shape
    latent = tf.keras.layers.Flatten()(inputs)
    latent = tf.keras.layers.Dense(feature_number, name='features')(latent)
    latent = tf.keras.layers.Dense(model_width * shape[1] * shape[2])(latent)
    latent = tf.keras.layers.Reshape((shape[1], shape[2], model_width))(latent)
    return latent

### Part 5.1 — UNet full architecture

In [ ]:
def UNet(length=64, width=64, model_depth=5, num_channel=2, model_width=32, kernel_size=3,
         ae=0, feature_number=1024, M=256):
    """Encoder-decoder UNet with skip connections (paper Section III-A)."""
    if length == 0 or model_depth == 0 or model_width == 0 or num_channel == 0 or kernel_size == 0:
        raise ValueError('Please check the values of the input parameters!')

    convs = {}
    inputs = tf.keras.Input((length, width, num_channel))
    pool = inputs

    for i in range(1, model_depth + 1):
        conv = Conv_Block(pool, model_width, kernel_size, 2 ** (i - 1))
        conv = Conv_Block(conv, model_width, kernel_size, 2 ** (i - 1))
        pool = tf.keras.layers.MaxPooling2D(pool_size=(2, 2))(conv)
        convs[f'conv{i}'] = conv

    if ae == 1:
        pool = Feature_Extraction_Block(pool, model_width, feature_number)

    conv = Conv_Block(pool, model_width, kernel_size, 2 ** model_depth)
    conv = Conv_Block(conv, model_width, kernel_size, 2 ** model_depth)

    deconv = conv
    convs_list = list(convs.values())
    for j in range(model_depth):
        skip = convs_list[model_depth - j - 1]
        deconv = trans_conv2D(deconv, model_width, 2 ** (model_depth - j - 1))
        deconv = Concat_Block(deconv, skip)
        deconv = Conv_Block(deconv, model_width, kernel_size, 2 ** (model_depth - j - 1))
        deconv = Conv_Block(deconv, model_width, kernel_size, 2 ** (model_depth - j - 1))

    deconv = tf.keras.layers.Conv2DTranspose(16, (2, 2), strides=(2, 2), padding='same')(deconv)
    h = tf.keras.layers.Conv2DTranspose(16, (2, 2), strides=(2, 2), padding='same')(inputs)
    deconv = Concat_Block(deconv, h)
    deconv = Conv_Block(deconv, model_width, kernel_size, 16)

    if M == 256:
        outputs = tf.keras.layers.Conv2DTranspose(1, (2, 2), activation='linear', strides=(2, 2), padding='same')(deconv)
    elif M == 512:
        outputs = tf.keras.layers.Conv2DTranspose(1, (4, 4), strides=(4, 4), padding='same')(deconv)
    else:
        raise ValueError('M must be 256 or 512')

    return tf.keras.Model(inputs=[inputs], outputs=[outputs])

In [ ]:
unet_model = UNet(M=256)
unet_model.summary()

## Part 6 — Model architecture: ResNet

In [ ]:
def res_conv(x, filters=12):
    """Residual block: two Conv2D+BN layers with a skip-add (paper Section III-B)."""
    x_skip = x
    x = Conv2D(filters, kernel_size=(5, 5), strides=(1, 1), padding='same')(x)
    x = BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    x = Conv2D(filters, kernel_size=(5, 5), strides=(1, 1), padding='same')(x)
    x = BatchNormalization()(x)
    x = Add()([x, x_skip])
    x = tf.keras.layers.Activation('relu')(x)
    return x

In [ ]:
def Resnet(input_shape=(64, 64, 2), output_dim=1, num_res_blocks=64):
    """64-layer residual super-resolution network."""
    x_in = Input(shape=input_shape)
    x = tf.keras.layers.Conv2DTranspose(12, (5, 5), strides=(2, 2), padding='same')(x_in)
    for _ in range(num_res_blocks):
        x = res_conv(x)
    x = tf.keras.layers.Conv2DTranspose(output_dim, (5, 5), strides=(2, 2), padding='same')(x)
    return Model(inputs=x_in, outputs=x)

In [ ]:
resnet_model = Resnet(input_shape=(64, 64, 2))
resnet_model.summary()

## Part 7 — Model compile (paper Table I hyperparameters)

In [ ]:
unet_model.compile(optimizer=Adam(learning_rate=0.001), loss='mse')
print('UNet compiled: Adam(lr=0.001), loss=mse')

In [ ]:
resnet_model.compile(optimizer=RMSprop(learning_rate=0.003), loss='mse')
print('ResNet compiled: RMSprop(lr=0.003), loss=mse')

## Part 8 — `tf.data` pipeline toiri koro loaded array theke

In [ ]:
BATCH_SIZE = 32  # paper Table I

if X_train is not None:
    train_ds = tf.data.Dataset.from_tensor_slices((X_train, Y_train))
    train_ds = train_ds.shuffle(buffer_size=min(len(X_train), 2000), seed=42)
    train_ds = train_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    print('train_ds ready, batches per epoch =', -(-len(X_train) // BATCH_SIZE))  # ceil division
else:
    train_ds = None

In [ ]:
if X_val is not None:
    val_ds = tf.data.Dataset.from_tensor_slices((X_val, Y_val))
    val_ds = val_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    print('val_ds ready, batches =', len(X_val) // BATCH_SIZE + 1)
else:
    val_ds = None

## Part 9 — Training (paper Table I: batch=32, epochs=500)

**Shabdhan**: paper-er training generator *infinite* — protiti epoch-e notun random sample toiri hoy.
Ekhane amra tomar **fixed, saved** dataset (`X_train`) use korchi, tai eta paper-er exact setup na —
protiti epoch-e shei-i 10000 (ba jotogula tumi generate korecho) sample repeat hobe. Beshi epoch dile
overfitting-er shombhabona ache; dorkar hole `EPOCHS` komiye/barhiye dekho.

`EPOCHS` niche choto rakha ache (druto test-er jonno) — full paper reproduction-er jonno `EPOCHS = 500`
kore dao (kintu Colab-e onek shomoy nebe, GPU runtime use koro).

In [ ]:
EPOCHS = 5  # <-- paper-er value: 500. Druto test-er jonno choto rakha ache, change kore dekho.

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        os.path.join(MODEL_DIR, 'unet_trained_best.weights.h5'),
        save_best_only=True, save_weights_only=True, monitor='val_loss',
    ),
]

In [ ]:
if train_ds is not None:
    unet_history = unet_model.fit(
        train_ds,
        epochs=EPOCHS,
        validation_data=val_ds,
    )
else:
    unet_history = None
    print('X_train paoa jayni — UNet training skip.')

In [ ]:
if unet_history is not None:
    plt.figure(figsize=(6, 4))
    plt.plot(unet_history.history['loss'], label='train loss')
    if 'val_loss' in unet_history.history:
        plt.plot(unet_history.history['val_loss'], label='val loss')
    plt.xlabel('Epoch'); plt.ylabel('MSE loss'); plt.title('UNet training curve')
    plt.legend(); plt.grid(True); plt.show()

In [ ]:
callbacks_resnet = [
    tf.keras.callbacks.ModelCheckpoint(
        os.path.join(MODEL_DIR, 'resnet_trained_best.weights.h5'),
        save_best_only=True, save_weights_only=True, monitor='val_loss',
    ),
]

if train_ds is not None:
    resnet_history = resnet_model.fit(
        train_ds,
        epochs=EPOCHS,
        validation_data=val_ds,
    )
else:
    resnet_history = None
    print('X_train paoa jayni — ResNet training skip.')

In [ ]:
if resnet_history is not None:
    plt.figure(figsize=(6, 4))
    plt.plot(resnet_history.history['loss'], label='train loss')
    if 'val_loss' in resnet_history.history:
        plt.plot(resnet_history.history['val_loss'], label='val loss')
    plt.xlabel('Epoch'); plt.ylabel('MSE loss'); plt.title('ResNet training curve')
    plt.legend(); plt.grid(True); plt.show()

In [ ]:
if unet_history is not None:
    unet_model.save_weights(os.path.join(MODEL_DIR, 'unet_trained_final.weights.h5'))
    print('Saved:', os.path.join(MODEL_DIR, 'unet_trained_final.weights.h5'))

if resnet_history is not None:
    resnet_model.save_weights(os.path.join(MODEL_DIR, 'resnet_trained_final.weights.h5'))
    print('Saved:', os.path.join(MODEL_DIR, 'resnet_trained_final.weights.h5'))

## Part 10 — (Alternative) Pretrained weights load koro

Training skip kore shorashori testing-e jete chaile — `inf_model_007_256_resnet.h5` ebong
`inf_model_007_256_unet.h5` (ba upore training theke toiri `*_trained_final.h5`) `MODEL_DIR`-e upload
kore ei cell run koro.

In [ ]:
RESNET_WEIGHTS = os.path.join(MODEL_DIR, 'inf_model_007_256_resnet.h5')
UNET_WEIGHTS = os.path.join(MODEL_DIR, 'inf_model_007_256_unet.h5')

if os.path.exists(RESNET_WEIGHTS):
    resnet_model.load_weights(RESNET_WEIGHTS)
    print('Loaded pretrained ResNet weights from', RESNET_WEIGHTS)
else:
    print(f'{RESNET_WEIGHTS} paoa jayni — ei model-er weight already-trained/loaded-i dhore nichi.')

if os.path.exists(UNET_WEIGHTS):
    unet_model.load_weights(UNET_WEIGHTS)
    print('Loaded pretrained UNet weights from', UNET_WEIGHTS)
else:
    print(f'{UNET_WEIGHTS} paoa jayni — ei model-er weight already-trained/loaded-i dhore nichi.')

## Part 11 — Inference utility functions (blob detection, angle conversion, metrics)

Paper-er `TVT_Blob_Inference.py`-r protita function alada cell-e — model-er heatmap output theke peak
(blob) detect kore, seta angle-e convert kore, ground-truth-er shathe match kore, tarpor angular error
(RMSE) ebong detection probability (Pd) hishab kore.

In [ ]:
def prepare_prediction_for_peaks(prediction):
    """Model output ke 8-bit normalized image-e convert kore (blob detector-er jonno)."""
    prediction_map = prediction[:, :, 0].numpy()
    img_norm = cv2.normalize(prediction_map, None, 0, 255, cv2.NORM_MINMAX)
    return img_norm.astype(np.uint8)

In [ ]:
def get_blob_detector():
    """OpenCV SimpleBlobDetector, paper-er default parameter diye."""
    params = cv2.SimpleBlobDetector_Params()
    params.filterByColor = True
    params.blobColor = 255
    params.minThreshold = 0
    params.maxThreshold = 255
    params.filterByArea = True
    params.minArea = 1
    params.maxArea = 1000
    params.filterByCircularity = False
    params.filterByConvexity = False
    params.filterByInertia = False
    return cv2.SimpleBlobDetector_create(params)

detector = get_blob_detector()

In [ ]:
def reorder_keypoints(keypoints, img_norm):
    """Detected blob-gulo ke intensity onujayi descending order-e shajay."""
    coords = np.array([kp.pt for kp in keypoints])
    if len(coords) == 0:
        return [], np.array([])
    coords_rounded = np.round(coords).astype(int)
    amplitudes = []
    for (x, y) in coords_rounded:
        if 0 <= y < img_norm.shape[0] and 0 <= x < img_norm.shape[1]:
            amplitudes.append(img_norm[y, x])
        else:
            amplitudes.append(0)
    amplitudes = np.array(amplitudes)
    order = np.argsort(-amplitudes)
    return [keypoints[i] for i in order], amplitudes[order]

In [ ]:
def get_blob_peaks(pred, detector, use_threshold=False, threshold_value=50):
    """Model prediction theke peak (x, y) coordinate-gulo বের kore, amplitude shoho."""
    img_norm = prepare_prediction_for_peaks(pred)
    img_proc = img_norm
    if use_threshold:
        _, img_proc = cv2.threshold(img_norm, threshold_value, 255, cv2.THRESH_BINARY)
    keypoints = detector.detect(img_proc)
    keypoints, amplitudes = reorder_keypoints(keypoints, img_norm)
    peaks = np.array([kp.pt for kp in keypoints]) if len(keypoints) > 0 else np.zeros((0, 2))
    return peaks, amplitudes

In [ ]:
def wrap_2pi_to_minus_pi(a):
    a = np.asarray(a)
    return np.where(a > np.pi, a - 2 * np.pi, a)

def peaks_to_angles(peaks, margin_factor=3.0, sigma=0.07, grid_size=256):
    """Grid pixel coordinate theke (psi, phi) angle-e convert kore."""
    if peaks.shape[0] == 0:
        return np.array([]), np.array([])
    margin = margin_factor * sigma
    peaks_x_y = peaks.T
    extended_range = 2 * np.pi + 2 * margin
    freqs_ext = -margin + (peaks_x_y / grid_size) * extended_range
    freqs_minus_pi = wrap_2pi_to_minus_pi(freqs_ext)
    psi_est = np.arccos(-freqs_minus_pi[1] / np.pi)
    phi_est = np.arccos(freqs_minus_pi[0] / np.pi)
    return psi_est, phi_est

In [ ]:
def permute_pairs(A, B):
    """Hungarian algorithm diye A o B point set-er optimal (minimum distance) matching."""
    A = np.asarray(A); B = np.asarray(B)
    dist_matrix = np.linalg.norm(A[:, np.newaxis, :] - B[np.newaxis, :, :], axis=2)
    row_ind, col_ind = linear_sum_assignment(dist_matrix)
    return [(tuple(A[i]), tuple(B[j])) for i, j in zip(row_ind, col_ind)]

def prepare_for_metric(angles_est, feat):
    """Predicted angle-gulo ke ground-truth-er shathe Hungarian-match kore."""
    L = feat.shape[-1]
    if len(angles_est[0]) < L:
        psi_est = np.full((L,), np.nan)
        phi_est = np.full((L,), np.nan)
        psi_true, phi_true = feat[0], feat[1]
    else:
        angles_est = (angles_est[0][:L], angles_est[1][:L])
        pairs_est = list(zip(angles_est[0], angles_est[1]))
        pairs_true = list(zip(feat[0], feat[1]))
        permuted = permute_pairs(pairs_true, pairs_est)
        first = [p[0] for p in permuted]
        second = [p[1] for p in permuted]
        psi_true, phi_true = zip(*first)
        psi_est, phi_est = zip(*second)
    return np.array([psi_true, phi_true]), np.array([psi_est, phi_est])

In [ ]:
def get_ang_difference(gt_angles, pred_angles):
    """Circular angular difference, degree-e."""
    H = np.angle(np.exp(1j * gt_angles) * np.exp(-1j * pred_angles))
    return (H * (180 / np.pi)).flatten()

def filter_angles(ang_dif_flat, max_deg_error=1.0):
    """1-degree threshold diye 'good' (detected) vs 'bad' (missed) angle-e bhag kore."""
    good = ang_dif_flat[np.abs(ang_dif_flat) <= max_deg_error]
    bad = ang_dif_flat[np.abs(ang_dif_flat) > max_deg_error]
    return good, bad

## Part 12 — Ekta single test example diye pura inference stage-by-stage dekho

Eta shobcheye kajer cell jodi tumi bujhte chao model output theke angle-e kivabe pouchay — input,
ground truth, model prediction, detected peak shob ekshathe dekhabe.

In [ ]:
def run_single_example(model, X, Y, feat, idx, sigma=0.07, grid_size=256, model_name=''):
    data = X[idx]
    gt = Y[idx]
    pred = model(tf.expand_dims(data, axis=0), training=False)
    pred = tf.squeeze(pred, axis=0)

    peaks, amps = get_blob_peaks(pred, detector)
    L = feat.shape[-1] if feat is not None else len(peaks)
    order = np.argsort(-amps)
    peaks_sorted = peaks[order[:L]] if len(peaks) > 0 else peaks

    angles_est = peaks_to_angles(peaks_sorted, sigma=sigma, grid_size=grid_size)

    fig, axs = plt.subplots(1, 3, figsize=(13, 4))
    axs[0].imshow(data[:, :, 0], cmap='viridis'); axs[0].set_title('Input (real part)')
    axs[1].imshow(gt[:, :, 0], cmap='hot'); axs[1].set_title('Ground truth')
    axs[2].imshow(pred.numpy()[:, :, 0], cmap='hot'); axs[2].set_title(f'{model_name} prediction')
    if len(peaks_sorted) > 0:
        axs[2].scatter(peaks_sorted[:, 0], peaks_sorted[:, 1], c='cyan', marker='x', s=60)
    for ax in axs:
        ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout()
    plt.show()

    if feat is not None:
        gt_angles, pred_angles = prepare_for_metric(angles_est, feat)
        diffs = get_ang_difference(gt_angles, pred_angles)
        print(f'{model_name} — true (psi,phi) [rad]:\n', feat)
        print(f'{model_name} — estimated (psi,phi) [rad] (matched order):\n', pred_angles)
        print(f'{model_name} — angular error (deg): {diffs}')
    return peaks_sorted, angles_est

In [ ]:
if X_val is not None:
    idx = 0
    feat0 = feat_val[idx] if feat_val is not None else None
    print('=== ResNet ===')
    run_single_example(resnet_model, X_val, Y_val, feat0, idx, model_name='ResNet')
    print('=== UNet ===')
    run_single_example(unet_model, X_val, Y_val, feat0, idx, model_name='UNet')
else:
    print('X_val paoa jayni, single-example demo skip.')

## Part 13 — Pura test-set-e evaluation (RMSE + Pd per SNR condition)

**Shomoy shotorko**: pura test set (8 SNR x 1000 sample = 8000/model) CPU-e model-proti ~1-2 ghonta
nite pare (cv2 blob-detection python loop-er bottleneck). `MAX_EVAL_SAMPLES` diye choto kore druto test
kore dekhte paro, tarpor `None` kore pura set-e run koro.

In [ ]:
MAX_EVAL_SAMPLES = 200  # None dile pura test set use hobe (paper-scale run — onek shomoy nebe)

def run_inference_and_metrics(model, X, meta, feat, max_deg_error=1.0, max_samples=None, model_name=''):
    n = len(X) if max_samples is None else min(len(X), max_samples)
    results = {}
    for i in tqdm(range(n), desc=f'Evaluating {model_name}', unit='ej'):
        data = X[i]
        L, SNR, QP = int(meta[i, 0]), int(meta[i, 1]), int(meta[i, 2])
        condition = (L, SNR, QP)

        pred = model(tf.expand_dims(data, axis=0), training=False)
        pred = tf.squeeze(pred, axis=0)

        peaks, amps = get_blob_peaks(pred, detector)
        order = np.argsort(-amps)
        peaks = peaks[order[:L]] if len(peaks) > 0 else peaks

        angles_est = peaks_to_angles(peaks)
        gt_angles, pred_angles = prepare_for_metric(angles_est, feat[i])
        results.setdefault(condition, []).append({'gt': gt_angles, 'pred': pred_angles})

    final_rmse, final_pd = {}, {}
    for condition, examples in results.items():
        good_all = []
        for ex in examples:
            if np.isnan(ex['pred']).any():
                continue
            diffs = get_ang_difference(ex['gt'], ex['pred'])
            good, _ = filter_angles(diffs, max_deg_error=max_deg_error)
            good_all.append(good)
        good_all = np.concatenate(good_all) if good_all else np.array([])
        n_total = sum(len(ex['gt'][0]) for ex in examples)
        final_pd[condition] = len(good_all) / n_total if n_total > 0 else np.nan
        final_rmse[condition] = np.sqrt(np.mean(good_all ** 2)) if len(good_all) > 0 else np.nan

    return final_rmse, final_pd

### Part 13.1 — ResNet evaluation

In [ ]:
if X_test is not None:
    resnet_rmse, resnet_pd = run_inference_and_metrics(
        resnet_model, X_test, meta_test, feat_test,
        max_samples=MAX_EVAL_SAMPLES, model_name='ResNet',
    )
    for cond in sorted(resnet_rmse, key=lambda c: c[1]):
        print(cond, f'RMSE={resnet_rmse[cond]:.4f}', f'Pd={resnet_pd[cond]:.4f}')
else:
    resnet_rmse, resnet_pd = {}, {}
    print('X_test paoa jayni — ResNet evaluation skip.')

### Part 13.2 — UNet evaluation

In [ ]:
if X_test is not None:
    unet_rmse, unet_pd = run_inference_and_metrics(
        unet_model, X_test, meta_test, feat_test,
        max_samples=MAX_EVAL_SAMPLES, model_name='UNet',
    )
    for cond in sorted(unet_rmse, key=lambda c: c[1]):
        print(cond, f'RMSE={unet_rmse[cond]:.4f}', f'Pd={unet_pd[cond]:.4f}')
else:
    unet_rmse, unet_pd = {}, {}
    print('X_test paoa jayni — UNet evaluation skip.')

## Part 14 — Result plots (RMSE / Pd vs SNR)

In [ ]:
def plot_rmse_pd(rmse_dict, pd_dict, target_L=3, target_QP=16, title=''):
    snrs, rmses, pds = [], [], []
    for (L, SNR, QP), val in rmse_dict.items():
        if L == target_L and QP == target_QP:
            snrs.append(SNR); rmses.append(val); pds.append(pd_dict[(L, SNR, QP)])
    order = np.argsort(snrs)
    snrs = np.array(snrs)[order]; rmses = np.array(rmses)[order]; pds = np.array(pds)[order]

    fig, axs = plt.subplots(1, 2, figsize=(10, 4))
    axs[0].plot(snrs, rmses, marker='o'); axs[0].set_xlabel('SNR (dB)'); axs[0].set_ylabel('RMSE (deg)')
    axs[0].set_title(f'{title} RMSE vs SNR (L={target_L}, QP={target_QP})'); axs[0].grid(True)
    axs[1].plot(snrs, pds, marker='s', color='green'); axs[1].set_xlabel('SNR (dB)'); axs[1].set_ylabel('Pd')
    axs[1].set_title(f'{title} Pd vs SNR (L={target_L}, QP={target_QP})'); axs[1].set_ylim(0, 1.05); axs[1].grid(True)
    plt.tight_layout(); plt.show()
    return snrs, rmses, pds

In [ ]:
if resnet_rmse:
    resnet_snrs, resnet_rmses, resnet_pds = plot_rmse_pd(resnet_rmse, resnet_pd, title='ResNet')

In [ ]:
if unet_rmse:
    unet_snrs, unet_rmses, unet_pds = plot_rmse_pd(unet_rmse, unet_pd, title='UNet')

### Part 14.1 — ResNet vs UNet comparative plot

In [ ]:
if resnet_rmse and unet_rmse:
    fig, axs = plt.subplots(1, 2, figsize=(12, 4))
    axs[0].plot(resnet_snrs, resnet_rmses, marker='o', label='ResNet', color='blue')
    axs[0].plot(unet_snrs, unet_rmses, marker='o', label='UNet', color='orange')
    axs[0].set_xlabel('SNR (dB)'); axs[0].set_ylabel('RMSE (deg)'); axs[0].set_title('RMSE vs SNR')
    axs[0].grid(True); axs[0].legend()

    axs[1].plot(resnet_snrs, resnet_pds, marker='s', label='ResNet', color='blue')
    axs[1].plot(unet_snrs, unet_pds, marker='s', label='UNet', color='orange')
    axs[1].set_xlabel('SNR (dB)'); axs[1].set_ylabel('Pd'); axs[1].set_title('Pd vs SNR')
    axs[1].set_ylim(0, 1.05); axs[1].grid(True); axs[1].legend()
    plt.tight_layout(); plt.show()
else:
    print('Dutoi model-er result na thakle comparative plot hobe na.')

### Part 14.2 — Metric dictionary o figure gulo save koro

In [ ]:
OUTPUT_DIR = 'results'
os.makedirs(OUTPUT_DIR, exist_ok=True)

if resnet_rmse:
    with open(os.path.join(OUTPUT_DIR, 'resnet_rmse.pkl'), 'wb') as f:
        pickle.dump(resnet_rmse, f)
    with open(os.path.join(OUTPUT_DIR, 'resnet_pd.pkl'), 'wb') as f:
        pickle.dump(resnet_pd, f)

if unet_rmse:
    with open(os.path.join(OUTPUT_DIR, 'unet_rmse.pkl'), 'wb') as f:
        pickle.dump(unet_rmse, f)
    with open(os.path.join(OUTPUT_DIR, 'unet_pd.pkl'), 'wb') as f:
        pickle.dump(unet_pd, f)

print('Saved to', OUTPUT_DIR)
print(sorted(os.listdir(OUTPUT_DIR)) if os.path.exists(OUTPUT_DIR) else 'nothing saved')

## Sesh kotha

- Protiti Part ekta alada, self-contained cell-group — jekono ekta function/cell shudhu edit kore
  abar shei cell-tuku-i re-run korte paro, notebook-er baki part touch korte hobe na.
- `HOLDOUT_FRACTION`, `EPOCHS`, `BATCH_SIZE`, `MAX_EVAL_SAMPLES` — ei charta variable-i shobcheye
  common jinish ja tumi change korte chaite paro.
- Training-e (Part 9) paper-er infinite-generator behavior nei (fixed dataset repeat hocche
  protiti epoch-e) — eta accept na hole nijer moto ekta `tf.data.Dataset.from_generator` diye
  fresh-sample-per-epoch pipeline banate paro, kintu tar jonno original steering-vector/beamforming/
  channel-generation code (dldoa_dataset_generation.py) abar lagbe.